## Part 09. 프로젝트 루트 확인과 4개 CSV 불러오기

In [149]:
# 35. 프로젝트 루트 설정
from pathlib import Path
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_dir = project_root / "data" / "raw"
print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: c:\dev\ai-data-analysis
데이터 폴더: c:\dev\ai-data-analysis\data\raw
데이터 폴더 존재: True


In [150]:
# 36. pandas와 CSV 불러오기
# 파일을 데이터 프레임이라고 함. 데이터 프레임만 가지고 있고 프린트하지 않으면 화면에 표시 안됨
import pandas as pd
customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")

In [151]:
# 37. 기본 구조와 주요 키 확인
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}
for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (301, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (765, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [152]:
key_checks = {
    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],
}

for name, series in key_checks.items():
    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),
    )


customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


## Part 10. 컬럼 선택, 조건 필터링, 정렬

In [153]:
# 38. Series와 DataFrame 선택
# series가 여러개 모이면 그게 dataframe임.
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]
print(type(city_series))
print(type(customer_view))
display(customer_view.head())

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


In [154]:
## 39. 단일 조건 필터링
customers_over_30 = customers[
    customers["age"] >= 30
]
print(len(customers), len(customers_over_30))
display(customers_over_30.head())

150 111


,customer_id,name,gender,age,city,signup_date
1,2,김정호,F,32,대구,2025-11-30
2,3,이경수,F,61,성남,2024-07-10
3,4,조영호,F,55,울산,2026-05-11
5,6,김지원,F,32,성남,2026-07-25
6,7,이상현,F,53,인천,2025-01-09


In [155]:
# 40. 복합 조건 필터링
# & 와 or 조건 이용해서 복합조건 가능
#30세 이상이면서 서울 거주:
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]
display(seoul_over_30.head())

#서울 또는 부산:
seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]
display(
    seoul_or_busan["city"].value_counts()
)

#완료 주문이 아닌 주문:
# ~표시는 어떤 조건을 부정하는 것. completed가 아닌 주문만 필터링
# 그래서 canceled, refunded가 나옴
not_completed = orders[
    ~(orders["order_status"] == "completed")
]
display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)


,customer_id,name,gender,age,city,signup_date
8,9,송지민,M,69,서울,2025-11-16
14,15,장정식,M,69,서울,2026-07-02
29,30,이민재,F,32,서울,2023-08-11
47,48,김예은,F,47,서울,2025-04-29
65,66,김재호,F,39,서울,2025-12-31


city
부산    16
서울    15
Name: count, dtype: int64

order_status
cancelled    65
refunded     52
Name: count, dtype: int64

In [156]:
# 41. 상품 가격 정렬
# 기본은 5개인데 haed(10)으로 바꿔서 10개만 보여주기
# sort_values()로 정렬 가능. ascending=False로 내림차순 정렬
# 들여쓰기 잘못하면 실행 안됨
expensive_products = (
    products
    .sort_values("price", ascending=False)
    .head(10)
)
display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


## Part 11. line_total 생성과 전체 주문 금액 구분

In [157]:
# 42. 작업용 복사본과 파생 컬럼
# 원본을 보존하기 위해서 이렇게 copy()로 복사해서 작업용으로 사용. 원본은 그대로 보존됨
order_items_work = order_items.copy()

# dataframe에 새로운 컬럼을 추가할 때는 이렇게 새로운 컬럼 이름을 지정하고 계산식을 넣으면 됨
# quantity * unit_price 해서 line_total 컬럼을 새로 만들어서 추가
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)

In [158]:
# 결과 확인
display(
    order_items_work[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ]
    ].head()
)

# 위에 코드를 언제 다 이쁘게 정리하겠어. 그냥 아래 코드로 간단하게 찍어보면 됨.
print(order_items_work.head())

,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


   order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000
3              4         1           9         3      193000      579000
4              5         2          72         4      189000      756000


In [159]:
# 43. 수작업 검증
# loc와 iloc는 인덱스로 접근하는 방법. loc는 라벨로 접근, iloc는 위치로 접근. 특정 부분만 잘라낼때 사용함
sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

수작업: 306000
파생 컬럼: 306000
일치: True


In [160]:
# 44. 전체 주문상세 금액
all_order_amount = order_items_work["line_total"].sum()
print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 255770000


In [161]:
# 45. 병합용 주문 컬럼 선택
orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()

print(orders_for_merge.shape)
print(orders_for_merge.head())

(301, 4)
   order_id  customer_id  order_date order_status
0         1          123  2026-06-04    completed
1         2           77  2025-08-20    cancelled
2         3          138  2025-12-17    cancelled
3         4           57  2026-02-27    cancelled
4         5          125  2026-01-18    cancelled


In [162]:
# 46. 주문상세와 주문 병합
# 주문이 부모고 주문 상세가 자식임. 부모 없는 자식도 있음 (고아데이터. 이건 못씀)
# order_items_work는 order에 파생컬럼이 추가됨. line_total이 추가됨. 단가*수량.
order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)

In [163]:
# 47. 병합 검증
print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

병합 전 행 수: 765
병합 후 행 수: 765


order_match
both          764
left_only       1
right_only      0
Name: count, dtype: int64

In [164]:
# 미매칭 확인:
unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match
764,765,999,101,5,32000,160000,NaN,NaN,NaN,left_only


In [165]:
# 48. 완료 주문 분석셋
# vluaue_counts는 특정 컬럼의 값이 몇 번 나왔는지 세어주는 함수.
display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)

order_status
completed    474
cancelled    162
refunded     128
NaN            1
Name: count, dtype: int64

In [166]:
completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [167]:
# Print는 검증하는 것임. 
# 상세행은 영수증에 들어간 모든 제품.
print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)
print(
    "완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)
print(
    "완료 주문 매출:",
    completed_sales["line_total"].sum(),
)

완료 주문상세 행: 474
완료 주문 수: 184
완료 주문 고객 수: 100
완료 주문 매출: 148990000


## Part 13. 상품 데이터 병합과 카테고리·상품 매출

In [168]:
# 49. 필요한 상품 정보만 선택 (컬럼을 말하는거임)
products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()

print (products_for_merge.head())

   product_id product_name category
0           1  전자기기 상품 001     전자기기
1           2    도서 상품 002       도서
2           3  전자기기 상품 003     전자기기
3           4  생활용품 상품 004     생활용품
4           5    식품 상품 005       식품


In [169]:
# 50. 완료 주문상세와 상품 병합
completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [170]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

474 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64

In [171]:
# 51. 카테고리별 매출
# 여기서 '별'자가 보이면 grouping 집계를 해야 하는구나 생각하기
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
3,스포츠,31743000,85,67,295,100
5,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
1,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42
0,도서,16389000,52,46,149,58
6,패션,10587000,33,27,111,37


In [172]:
#52. 카테고리 합계 검증
# 합계가 다르면 카테고리 결측, 상품 미매칭, 중복 병합과 필터 범위 차이를 확인한다.
category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

148990000
148990000
True


In [173]:
# 53. 상품별 매출
# agg 함수는 전체 판매금액임. 
# unique를 쓰지 않는다면 동일 고객이 2번 구매했다면 카운트 안됨.
# .sort 는 정렬
# 판매량 상위와 매출 상위는 다를 수 있으므로 두 기준을 별도로 비교합니다.
product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
39,41,스포츠 상품 041,스포츠,5705000,35,12,11
11,12,식품 상품 012,식품,4375000,25,7,7
8,9,스포츠 상품 009,스포츠,3860000,20,6,5
70,72,뷰티 상품 072,뷰티,3780000,20,6,6
69,71,전자기기 상품 071,전자기기,3703000,23,5,5
66,68,스포츠 상품 068,스포츠,3640000,26,8,8
78,81,전자기기 상품 081,전자기기,3630000,22,6,6
10,11,패션 상품 011,패션,3565000,31,7,7
20,22,생활용품 상품 022,생활용품,3248000,29,8,8
86,89,생활용품 상품 089,생활용품,3090000,30,11,11


# Part 14. 월별 매출과 고객별 구매 금액

In [174]:
# 54. 주문 날짜 변환과 주문 월 생성
# errors="coerce"는 변환할 수 없는 갑사을 오류로 중단하지 말고 결측값(NaN, NaT등)으로 바꾸라는 의미. 
# 결측이 되면 true임. true는 1, false는 0임. 다합치면 갯수임. 
completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

날짜 변환 실패: 0


In [175]:
completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

In [176]:
completed_items.head()

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match,product_name,category,product_match,order_month
0,1,1,100,3,102000,306000,123.0,2026-06-04,completed,both,도서 상품 100,도서,both,2026-06
1,2,1,87,5,25000,125000,123.0,2026-06-04,completed,both,도서 상품 087,도서,both,2026-06
2,3,1,7,3,142000,426000,123.0,2026-06-04,completed,both,도서 상품 007,도서,both,2026-06
3,4,1,9,3,193000,579000,123.0,2026-06-04,completed,both,스포츠 상품 009,스포츠,both,2026-06
4,13,6,83,3,24000,72000,87.0,2026-04-18,completed,both,전자기기 상품 083,전자기기,both,2026-04


In [177]:
# 55. 월별 매출
# 연도만 있었는데, order_month 월별 합계
monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)
display(monthly_sales)

,order_month,total_sales,order_count,customer_count,quantity_sold
0,2025-08,5869000,8,8,52
1,2025-09,16147000,19,19,132
2,2025-10,12385000,15,15,120
3,2025-11,23550000,24,23,233
4,2025-12,9876000,13,13,99
5,2026-01,10851000,13,13,105
6,2026-02,16504000,21,20,150
7,2026-03,9885000,18,16,102
8,2026-04,15536000,17,16,157
9,2026-05,15310000,19,18,152


In [178]:
# 56. 고객별 구매 금액
customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)

In [179]:
print(customer_sales.head())

   customer_id  total_sales  order_count  quantity_sold
0          3.0      3178000            2             26
1          4.0       603000            1              6
2          5.0      2004000            2             13
3          6.0      1349000            2             14
4          7.0      1173000            1              9


In [180]:
print(customer_sales.columns)

Index(['customer_id', 'total_sales', 'order_count', 'quantity_sold'], dtype='str')


In [181]:
# 많이 구매한 고객 10명 알기
print(customer_sales.sort_values("total_sales", ascending=False).head(10))

    customer_id  total_sales  order_count  quantity_sold
76        117.0      4100000            5             48
62        102.0      3996000            4             35
51         83.0      3880000            4             39
21         30.0      3590000            5             32
29         40.0      3523000            4             27
13         20.0      3191000            2             25
0           3.0      3178000            2             26
70        111.0      3153000            3             38
42         66.0      3093000            4             30
97        147.0      2990000            2             21


In [182]:
# 57. 고객 속성 연결
# 개인정보 최소화를 위해 이름은 제외합니다.
customer_attributes = customers[
    ["customer_id", "gender", "age", "city"]
].copy()

In [183]:
customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values("total_sales", ascending=False)
)

In [184]:
display(
    customer_sales_detail["customer_match"].value_counts(
        dropna=False
    )
)
display(customer_sales_detail.head(10))

customer_match
both          100
left_only       0
right_only      0
Name: count, dtype: int64

,customer_id,total_sales,order_count,quantity_sold,gender,age,city,customer_match
76,117.0,4100000,5,48,F,65,성남,both
62,102.0,3996000,4,35,M,60,고양,both
51,83.0,3880000,4,39,F,22,수원,both
21,30.0,3590000,5,32,F,32,서울,both
29,40.0,3523000,4,27,M,23,서울,both
13,20.0,3191000,2,25,F,20,인천,both
0,3.0,3178000,2,26,F,61,성남,both
70,111.0,3153000,3,38,F,41,광주,both
42,66.0,3093000,4,30,F,39,서울,both
97,147.0,2990000,2,21,M,19,부산,both


## Part 15. 결과 CSV 저장과 반복 점검 함수


In [185]:
## 58. 결과 폴더 생성
output_dir = project_root / "reports" / "chapter04"
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)


c:\dev\ai-data-analysis\reports\chapter04


In [186]:
# 59. 결과 파일 저장
outputs = {
    "category_sales.csv": category_sales,
    "product_sales.csv": product_sales,
    "monthly_sales.csv": monthly_sales,
    "customer_sales.csv": customer_sales_detail,
}
for file_name, df in outputs.items():
    output_path = output_dir / file_name
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(
        file_name,
        output_path.exists(),
        output_path.stat().st_size,
    )

category_sales.csv True 309
product_sales.csv True 4833
monthly_sales.csv True 415
customer_sales.csv True 3648


In [187]:
# 60. 저장 결과 다시 읽기
# 저장 후 다시 읽어 컬럼, 행 수와 한글 표시가 유지되는지 확인합니다.
saved_category_sales = pd.read_csv(
    output_dir / "category_sales.csv"
)
display(saved_category_sales.head())
print(saved_category_sales.shape)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
0,스포츠,31743000,85,67,295,100
1,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
3,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42


(7, 6)


In [188]:
# 61. 병합 점검 함수
def check_merge_result(
    *,
    name: str,
    left_rows: int,
    merged: pd.DataFrame,
    indicator_column: str,
) -> None:
    print(f"[{name}]")
    print("병합 전 행 수:", left_rows)
    print("병합 후 행 수:", len(merged))
    print(
        merged[indicator_column].value_counts(
            dropna=False
        )
    )

In [189]:
# 위에 만든 함수 호출
check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)

[주문상세-주문]
병합 전 행 수: 765
병합 후 행 수: 765
order_match
both          764
left_only       1
right_only      0
Name: count, dtype: int64


In [190]:
# 62. 집계 합계 검증 함수
def check_total(
    *,
    name: str,
    source_total: float,
    summary_total: float,
) -> None:
    difference = source_total - summary_total
    print(f"[{name}]")
    print("원본 합계:", source_total)
    print("요약 합계:", summary_total)
    print("차이:", difference)

In [191]:
check_total(
    name="카테고리별 매출",
    source_total=completed_items["line_total"].sum(),
    summary_total=category_sales["total_sales"].sum(),
)

[카테고리별 매출]
원본 합계: 148990000
요약 합계: 148990000
차이: 0


### 65. LLM 코드 검증표

| 검증 항목 | 확인 내용 | 결과 |
|---|---|---|
| DataFrame | 실제 코드에서 생성한 DataFrame 변수명을 정확하게 사용했는가? | **충족** — 원본 DataFrame은 `customers`, `products`, `orders`, `order_items`를 사용했다. 분석 과정에서는 `order_items_work`, `order_sales`, `completed_sales`, `completed_items`, `category_sales`, `product_sales`, `monthly_sales`, `customer_sales`, `customer_sales_detail`을 실제 변수명과 일치하게 사용했다. |
| 컬럼 | 각 DataFrame에 실제로 존재하는 컬럼을 사용했는가? | **충족** — `quantity`, `unit_price`, `order_id`, `product_id`, `customer_id`, `order_date`, `order_status`, `product_name`, `category` 등 실제 컬럼을 사용했다. `line_total`, `total_sales`, `order_count`, `quantity_sold` 등의 집계 컬럼도 코드에서 정상적으로 생성했다. |
| 상태값 | 완료 주문을 선택할 때 실제 상태값인 `completed`를 사용했는가? | **충족** — `order_sales["order_status"] == "completed"` 조건을 사용했다. 완료 주문상세 474행, 고유 주문 184건, 고객 100명이 추출되었다. |
| 계산식 | 주문상품별 매출을 `quantity × unit_price`로 계산했는가? | **충족** — `quantity * unit_price`로 `line_total`을 계산했다. 첫 번째 행도 `3 × 102,000 = 306,000`으로 수작업 계산값과 일치했다. |
| 분석 범위 | 취소·환불 주문을 제외하고 완료 주문만 분석에 포함했는가? | **충족** — `completed_sales`와 `completed_items`를 기준으로 집계하여 `cancelled`와 `refunded` 주문을 매출 분석에서 제외했다. |
| 주문 수 | 주문상세 행 수가 아닌 고유 주문 수를 구하기 위해 `nunique()`를 사용했는가? | **충족** — `order_count=("order_id", "nunique")`를 사용했다. 완료 주문상세 474행에서 실제 고유 주문은 184건으로 계산되었다. |
| 병합 키 | DataFrame의 관계에 맞는 병합 키를 사용했는가? | **충족** — 주문상세–주문은 `order_id`, 주문상세–상품은 `product_id`, 고객별 매출–고객 속성은 `customer_id`로 병합했다. |
| `validate` | 병합 관계에 맞게 `many_to_one` 또는 `one_to_one`을 적용했는가? | **충족** — 주문상세–주문과 주문상세–상품에는 `many_to_one`을 적용했다. 고객별 매출–고객 속성에는 관계에 맞게 `one_to_one`을 적용했다. |
| `indicator` | 병합 후 미매칭 데이터를 확인하기 위해 `indicator`를 사용했는가? | **충족** — `order_match`, `product_match`, `customer_match`를 생성했다. 상품 474행과 고객 100행은 모두 매칭되었으며, 주문 병합에서는 미매칭 주문상세 1행을 확인했다. |
| 행 수 | 병합 전후 행 수를 비교하여 중복 증가나 누락을 확인했는가? | **충족** — 주문 병합은 765행에서 765행, 상품 병합은 474행에서 474행으로 유지되었다. 병합으로 인한 행 수 증가는 없었다. |
| 합계 | 원본 완료 주문 매출과 요약 결과의 매출 합계를 비교했는가? | **충족** — 원본 합계와 카테고리별 요약 합계가 모두 `148,990,000원`이었으며 차이는 `0원`이었다. |
| 개인정보 | 분석에 불필요한 고객 개인정보를 제외했는가? | **충족** — `customer_id`, `gender`, `age`, `city`만 사용하고 고객 이름인 `name`과 `signup_date`는 결과에서 제외했다. |